# Lab 10d: the LLM as scenario writer

**AI-powered Investment and Risk Management (25882), Lecture 10: AI in Risk Forecasting and Stress Testing**

Large language models can write stress narratives quickly and in bulk, and attach numbers to them. The lecture's *Narrate* slide listed what can go wrong: invented magnitudes, numbers that do not fit each other, sensitivity to the prompt, and no governance home yet. In 2026 the ECB found that turning narratives into consistent numbers was the weakest part of banks' own reverse stress tests, written by people.

This lab treats an LLM as a scenario writer and builds the checks it needs. Every scenario is asked four questions:

1. **Is it plausible?** How far is it from the months we have seen since 1970?
2. **Do its numbers fit each other?** Given nine of its ten returns, is the tenth what a crisis month would imply?
3. **Is it severe for the portfolio we hold?**
4. **How much did the wording of the prompt decide?**

**Scenarios.** By default the lab uses cached replies to two prompts that differ by one sentence, written once by Claude (Opus 5.5) in the authoring session and not edited after these checks were run. Set `USE_LIVE = True` to call the Claude API with your own key, or paste a JSON reply from any chatbot into `PASTE_JSON`.

**Data.** The ten-asset monthly panel of Labs 10a and 10b (eight US industries, long Treasuries, corporate bonds), 1970–2025.

**Runtime.** A few seconds offline; a minute or two with live calls.

## Setup

In [ ]:
import json
import os
import urllib.request

if not os.path.exists("airm10.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/VitaliAlexeev/AI_Investments_2026/main/airm10.py", "airm10.py")

import numpy as np
import pandas as pd
from IPython.display import display

import airm10 as A

## 1. Two prompts, one sentence apart

The prompts are identical except for the sentence describing severity. Both ask for JSON in a fixed schema, so the numbers can be checked mechanically.

In [ ]:
BODY = 'You are a risk analyst at an Australian superannuation fund that holds US assets. Write 8 distinct stress scenarios for a single month. {severity} For each scenario give a short name, a one-sentence narrative, and the total return over the month, in per cent, of each of these ten US assets: Banks, Insur (insurers), RlEst (real estate), Oil, Util (utilities), Hlth (health care), Softw (software), Rtail (retail), which are equity industry portfolios, plus Treasuries (long-term government bonds) and Corp bonds (investment-grade corporate bonds). Reply with JSON only: a list of objects with keys "name", "narrative" and "returns", where "returns" maps each of the ten asset keys to a number.'
PROMPTS = {
    "Severe but plausible": BODY.format(severity="Each should be severe but plausible: something that could "
                                                 "realistically happen within the next five years."),
    "Worst imaginable": BODY.format(severity="Each should be the worst month you can imagine for these assets."),
}

USE_LIVE = False             # True: call the Claude API (needs ANTHROPIC_API_KEY set in your environment)
MODEL = "claude-sonnet-5-5"  # any Claude model your key can use
PASTE_JSON = None            # or paste a JSON reply from any chatbot here, as a string

cache = A.load_cached_llm()
assert cache["prompts"] == PROMPTS, "The cached replies were made with different prompts; set USE_LIVE = True."
replies = {label: (A.parse_json_reply(A.call_claude(p, MODEL)) if USE_LIVE else cache["responses"][label])
           for label, p in PROMPTS.items()}
if PASTE_JSON:
    replies["Your prompt"] = A.parse_json_reply(PASTE_JSON)

frames = [A.scenario_frame(items, label) for label, items in replies.items()]
R = pd.concat([f[0] for f in frames])
narratives = pd.concat([f[1] for f in frames])
print(f"{len(R)} scenarios" + ("" if USE_LIVE else f" (cached: {cache['generated_by']})"))
display(narratives)
display(R)

## 2. Is it plausible?

Plausibility is measured as in Lab 10b: the Mahalanobis distance of the scenario's ten returns from the average month since 1970. The vertical lines mark the 99th percentile of historical months and the single most extreme one. As a second opinion, the two-regime mixture of Lab 10a scores each scenario's likelihood against every historical month.

<details>
<summary><b>The maths: two plausibility scores</b></summary>

With the 1970–2025 mean $\mu$ and covariance $\Sigma$, a scenario $x$ has distance $d(x)=\sqrt{(x-\mu)^\top\Sigma^{-1}(x-\mu)}$. A distance above the largest one in the data means no month since 1970 was as unusual *in this joint sense*. The mixture's log-density $\log p(x)$ is compared with $\log p(x_t)$ for every historical month $x_t$; the score reported is the share of historical months that were less likely than the scenario. Zero means the scenario is less likely than any month on record.
</details>

In [ ]:
X = A.asset_panel()
mu, S = X.mean().values, X.cov().values
d_hist = pd.Series(A.mahalanobis_rows(X.values, mu, S), index=X.index)
dist = pd.Series(A.mahalanobis_rows(R.values, mu, S), index=R.index)

mixture, _ = A.fit_mixture(X)
hist_ll = mixture.score_samples(X.values)
less_likely = pd.Series([np.mean(hist_ll < l) for l in mixture.score_samples(R.values)], index=R.index)

plausibility = pd.DataFrame({"Distance": dist, "Beyond every month since 1970": dist > d_hist.max(),
                             "Share of historical months less likely": less_likely})
A.distance_figure(dist, d_hist.max(), d_hist.quantile(0.99), "How far each scenario is from the months we have seen").show()
print(f"Most extreme month since 1970: {d_hist.idxmax()} (distance {d_hist.max():.1f}); 99th percentile {d_hist.quantile(0.99):.1f}")
display(plausibility.round(3))

Of the 8 *severe but plausible* scenarios, 0 lie beyond every month since 1970 (distance above 11.4, reached in 2009-04); of the 8 *worst imaginable* ones, 8 do. The mixture agrees in spirit: 0 and 6 scenarios respectively are less likely than every month on record.

Going beyond history is not a defect in a stress scenario: it is often the point. But it moves the burden of proof. A scenario beyond every month on record needs an explicit argument, as the Bank of England gave for its 1-in-100-year gilt shock after 2022.

## 3. Do the numbers fit each other?

A scenario is internally consistent if each number is roughly what the other nine imply. We judge that against the mixture's **volatile regime**, because the question is how assets move together in a crisis month, not in an average one. The heat map shows, for every scenario and asset, how many standard deviations the stated return is from what the other nine imply.

<details>
<summary><b>The maths: leave-one-out conditional z-scores</b></summary>

For $x \sim N(\mu, \Sigma)$ with precision matrix $Q = \Sigma^{-1}$ and $d = x - \mu$, the distribution of $x_j$ given all the other returns is normal with

$$\mathbb{E}[x_j \mid x_{-j}] = x_j - \frac{(Qd)_j}{Q_{jj}}, \qquad \operatorname{sd}(x_j \mid x_{-j}) = \frac{1}{\sqrt{Q_{jj}}},$$

so the conditional z-score is simply $z_j = (Qd)_j / \sqrt{Q_{jj}}$. We use the mean and covariance of the volatile regime and flag $|z_j| > 3$.
</details>

In [ ]:
volatile = int(np.argmax([np.trace(c) for c in mixture.covariances_]))
mu_v, S_v = mixture.means_[volatile], mixture.covariances_[volatile]
Z = A.conditional_z(R, mu_v, S_v)
implied = A.conditional_mean(R, mu_v, S_v)
A.z_heatmap(Z, "Each number against what the other nine imply in a crisis month (z-scores)").show()

flags = (Z.stack().rename("z").to_frame()
         .join(R.stack().rename("Stated (%)")).join(implied.stack().rename("Implied by the other nine (%)")))
flags = flags[flags["z"].abs() > 3].sort_values("z", key=np.abs, ascending=False)
display(flags.round(1))

2 of the 8 *severe but plausible* scenarios and 8 of the 8 *worst imaginable* ones contain at least one number more than three standard deviations from what the other nine imply in a crisis month. The largest gap: *Global depression begins* puts Corp bonds at −18%, where the other nine numbers imply about 6.5% (z = −11.2).

A flag is a question, not a verdict. Corporate bonds have fallen while Treasuries rallied: in September 2008 (−8.6% against +1.1%) and in March 2020 (−3.5% against +6.2%). The direction can be defended from history; a gap of 33 percentage points between them in one month needs an argument.

## 4. Is it severe, and for whom?

Translate each scenario into the six benchmark portfolios of Lab 10a, and compare with each portfolio's worst month since 1970.

In [ ]:
P = A.benchmark_portfolios()
pnl = R @ P.T
worst = (X @ P.T).min()
display(pd.concat([pnl, worst.to_frame("Worst month since 1970").T]).round(1))
q01_6040 = float(np.quantile(X.values @ P.loc["60/40"].values, 0.01))
print(f"60/40: worst 1% of months since 1970 lose {q01_6040:.1f}% or more")

For the 60/40 portfolio, the *severe but plausible* scenarios return between −3.5% and −9.4% in the month, and the *worst imaginable* ones between −6.7% and −28.9%; the worst month since 1970 was −13.4%. 7 and 1 scenarios respectively are milder for 60/40 than its worst 1% of historical months (−9.2%): severe for some assets, but not a stress test for this portfolio.

## 5. How much did the prompt decide?

In [ ]:
by_prompt = pd.DataFrame({
    "Median distance": dist.groupby(level=0, sort=False).median(),
    "Beyond every month since 1970": plausibility["Beyond every month since 1970"].groupby(level=0, sort=False).sum(),
    "With a number |z| > 3": (Z.abs() > 3).any(axis=1).groupby(level=0, sort=False).sum(),
    "Median 60/40 return (%)": pnl["60/40"].groupby(level=0, sort=False).median(),
})
display(by_prompt.round(2))

One sentence moved the median distance from 5.3 to 16.5 and the median 60/40 return from −4.3% to −11.2%. Nothing else about the request changed.

## 6. The human challenge

The checks do not decide; they route each scenario to the right conversation. Numbers that do not fit each other go back for revision. A scenario beyond every month on record needs an argument. One that does not hurt the portfolio is not a stress test for it.

In [ ]:
consistent = (Z.abs() <= 3).all(axis=1)
plausible = ~plausibility["Beyond every month since 1970"]
severe = pnl["60/40"] <= q01_6040
verdict = np.select([~consistent, ~plausible, ~severe],
                    ["Revise the numbers", "Justify: beyond every month since 1970", "Too mild for this portfolio"],
                    default="Ready for review")
review = pd.DataFrame({"Consistent": consistent, "Inside history": plausible, "Severe for 60/40": severe,
                       "Verdict": verdict}, index=R.index)
display(review)
display(review.groupby(level=0, sort=False)["Verdict"].value_counts().unstack(fill_value=0))

*Severe but plausible*: 5 too mild for this portfolio, 2 revise the numbers, 1 ready for review. *Worst imaginable*: 8 revise the numbers. A first-pass LLM scenario set is a draft for a risk committee, not an output for a regulator.

## What to notice

1. **An LLM writes stories fluently; the numbers need checking.** Plausibility, consistency and severity can all be tested mechanically once the output has a fixed schema.
2. **The prompt is a model parameter.** One sentence changed the severity of every scenario. Report the prompt with the scenarios, and vary it.
3. **The checks route, people decide.** Beyond-history scenarios are allowed, and sometimes necessary, but they carry the burden of proof. That is the governance gap the lecture flagged: nobody yet owns the validation of an LLM-written scenario.

## Exercises

**Core 1.** Pick the scenario with the largest flagged z-score. Rewrite its numbers so that every |z| is below 3 while keeping its narrative, and rerun sections 2 to 6. How much did its distance change?

**Core 2.** Paste a JSON reply from any chatbot you use into `PASTE_JSON`, using the same prompt, and rerun. How do its scenarios compare with the cached ones?

**Stretch 1.** With an API key, set `USE_LIVE = True` and run each prompt three times. How much do the median distance and the median 60/40 return vary *between runs of the same prompt*, compared with *between prompts*?

**Stretch 2.** Add a third prompt that asks the model to calibrate each scenario to a named historical episode (for example October 2008 or March 2020). Does anchoring to history make the scenarios more consistent, and less severe?